In [47]:
import os
import sys

import json
import random
import pandas as pd

from pathlib import Path
from IPython.display import clear_output
from collections import Counter

In [2]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

from corpus import get_gt_dict
import api

In [22]:
EPI_NUM = "008"
CALL_API = False

error_categories = {
    "1": "Wrong relation type",
    "2": "Duplicate/coreference",
    "3": "Unsupported relation",
    "4": "Entity boundary/synonyms mismatch",
    "5": "Missing relation",
    "6": "Evaluation artefact",
    "7": "Other/unclear"
}

random.seed(42)
SAVE_PATH = f"../../data/results/error_codes"

In [4]:
with open(f"../../data/results/epis/biored_train/epi_{EPI_NUM}.json") as f:
    epi_log = json.load(f)

In [5]:
error_samples = {}

SAMPLE_SIZE = 50

for i in ("relations", "entities"):

    fp = epi_log["errors"]["errors"][i]["false_positives"]
    fn = epi_log["errors"]["errors"][i]["false_negatives"]

    error_samples[i] = {
        "fp": random.Random(42).sample(fp, SAMPLE_SIZE) if len(fp) > SAMPLE_SIZE else fp,
        "fn": random.Random(42).sample(fn, SAMPLE_SIZE) if len(fn) > SAMPLE_SIZE else fn,
    }

    print(f"'{i}' False Positive Sample Length: {len(error_samples[i]["fp"])}")
    print(f"'{i}' False Negative Sample Length: {len(error_samples[i]["fn"])}")
    print("")

'relations' False Positive Sample Length: 50
'relations' False Negative Sample Length: 50

'entities' False Positive Sample Length: 50
'entities' False Negative Sample Length: 13



In [6]:
# ---------- Import GTs ----------
gt = pd.read_csv("../../data/processed/biored/br_train_gt.csv")
gt_dict = get_gt_dict(gt)

In [7]:
# ---------- Import BioRED train abstracts ----------
abstracts = pd.read_csv("../../data/processed/biored/br_train.csv")

In [8]:
ERROR_SAMPLE_DIR = f"../../data/error_sampling"

os.makedirs(ERROR_SAMPLE_DIR, exist_ok=True)
with open(os.path.join(ERROR_SAMPLE_DIR, f"epi_{EPI_NUM}_error_samples.json"), "w") as f:
    json.dump(error_samples, f, indent=2, default=list)

In [9]:
client = api.create_client()

Created client:
 - Timeout: 60
 - Max retries: 0



In [28]:
if CALL_API:
    # ---------- Relations (LLM-categorised) ----------
    error_codes = []

    category_list_str = "\n".join(f"{k}: {v}" for k, v in error_categories.items())

    for error_type, errors in error_samples["relations"].items():
        for error in errors:

            pmid = error[0]

            matching_gts = [
                gt for gt in gt_dict["relations"]
                if gt[0] == pmid
            ]

            prompt = f"""You are reviewing a relation-extraction error made by an NLP system.\n\nError type: {error_type}\n- fp: a predicted relation that did not match the BioRED ground truth under the evaluation procedure.\n- fn: a BioRED ground-truth relation that was not matched by a prediction.\n\nRelation being analysed:\n{error}\n\nAbstract:\n{abstracts.set_index("pmid").loc[pmid]}\n\nGround-truth relations for this abstract:\n{chr(10).join(str(gt) for gt in matching_gts)}\n\nCategorise the primary cause of this error into exactly one of the categories below.\nUse the abstract and BioRED ground truth as the reference when making your decision.\nIf multiple categories could apply, select the category that best explains why the prediction and ground truth did not match.\n\nRespond with ONLY the category number, nothing else.\n\n{category_list_str}"""

            response = client.responses.create(
                model="gpt-5.6-luna",
                input=prompt
            )

            error_code = response.output_text.strip()

            error_codes.append({
                "error_type": str(error_type),
                "error": error,
                "category": error_categories.get(error_code, "Invalid")
            })

            # print(f"{error} -> {error_categories.get(error_code, 'Invalid')}")

            with open(f"{SAVE_PATH}/epi_{EPI_NUM}.json", "w") as f:
                json.dump(error_codes, f, indent=4)

            print(f"Saved `error_codes` for epi_{EPI_NUM} to `{SAVE_PATH}/epi{EPI_NUM}.json` successfully.")

else:
    with open(f"{SAVE_PATH}/epi_{EPI_NUM}.json", "r") as f:
        error_codes = json.load(f)

    print(f"Loaded `error_codes` for epi_{EPI_NUM} from `{SAVE_PATH}/epi{EPI_NUM}.json` successfully.")

Loaded `error_codes` for epi_008 from `../../data/results/error_codes/epi008.json` successfully.


In [ ]:
code_counts = Counter(error["category"] for error in error_codes)

In [45]:
code_counts

Counter({'Missing relation': 27,
         'Evaluation artefact': 23,
         'Entity boundary/synonyms mismatch': 20,
         'Unsupported relation': 18,
         'Wrong relation type': 11,
         'Duplicate/coreference': 1})